# 02 — Ingesta meteorológica (Open-Meteo)

Descarga clima diario 2017-2025 por distrito y lo cachea en `data/bronze/meteo/cache_meteo/` (local, ya no en Drive). Corre esta celda varias veces si Open-Meteo corta el batch por límite de rate.

In [1]:
import sys
from pathlib import Path

In [2]:
import pandas as pd
from denguekay.config.paths import REFERENCE
from denguekay.data.ingestion.openmeteo import download_pending_batch

distritos = pd.read_csv(REFERENCE / "distritos_piura_coords.csv")
print(len(distritos), "distritos")

65 distritos


In [3]:
import sys
print(sys.executable)

c:\Users\Rosa\RufoEsEternoElRegreso\Random\Dump\.venv\Scripts\python.exe


In [4]:
'''
Correr esta celda y luego resetear el kernel si es que sale algun relacionado con pyarrow. Si no, seguir con la siguiente celda.
'''

# %pip install --upgrade --force-reinstall pandas pyarrow

'\nCorrer esta celda y luego resetear el kernel si es que sale algun relacionado con pyarrow. Si no, seguir con la siguiente celda.\n'

In [5]:
# Corre esta celda repetidas veces hasta que diga 'Quedan: 0'
quedan = download_pending_batch(distritos)

Descargados: 65 | Pendientes: 0

En este batch: 0 | Quedan: 0
Listo: ya se puede pasar al armado de la data diaria (bronze -> silver).


## Estudio 06 · fase 1b · descargas aparte (no cambian lo de arriba)

Las celdas de arriba siguen descargando 2017-2025 con el modelo por defecto de Open-Meteo («Best match») en
`cache_meteo/`. Las de abajo escriben en **cachés separados**, así que no tocan esa descarga ni lo que lee
el notebook 03:

- **A · ERA5 fijo 2017-2025** → `data/bronze/meteo/cache_meteo_era5_2017_2025/`. Sirve para el notebook
  `research/06_model_improvement/notebooks/03_fase1b_consistencia_clima.ipynb`. Se corre primero.
- **B · historia 2000-2016** → `data/bronze/meteo/cache_meteo_2000_2016/`. Se corre **después** de que Rosa
  decida con qué modelo, a la vista de esa comparación.

Antes de correr A, confirmen el identificador del modelo: en https://open-meteo.com/en/docs/historical-weather-api
elijan ERA5 en «Reanalysis models» y miren el valor de `models=` en la URL que genera la página. Si no es
`era5`, cámbienlo en `MODELO_FIJO`. Si el API responde con columnas renombradas, `fetch_district` se detiene.
Igual que arriba: corran cada celda varias veces hasta que diga «Quedan: 0».


In [ ]:
from denguekay.config.paths import (
    CACHE_METEO_2000_2016, CACHE_METEO_ERA5_2017_2025, CACHE_METEO_ERA5_LAND_2017_2025,
)

MODELO_FIJO = "era5"  # ERA5, 0,25°, desde 1940; ERA5-Land no trae radiación solar según la documentación

# A · Corre esta celda repetidas veces hasta que diga 'Quedan: 0'
quedan_era5 = download_pending_batch(
    distritos, models=MODELO_FIJO, cache_dir=CACHE_METEO_ERA5_2017_2025,
)


**A2 · ERA5-Land fijo 2017-2025, muestra de 16 distritos** → `data/bronze/meteo/cache_meteo_era5_land_2017_2025/`.
Igual que A, con `models="era5_land"` (0,1°, unos 11 km), pero solo para una muestra: la descarga completa tarda
horas por el límite del API. Sirve para ver si ERA5-Land distingue la precipitación entre distritos vecinos antes de
decidir la fuente de 2000-2025. Caché propio: no toca `cache_meteo/` ni el silver vigente.

- **5 grupos completos que comparten celda en ERA5 (0,25°)**, elegidos donde se concentran los casos:
  Piura, Bellavista y Miguel Checa (43, 51, 55); La Arena, La Unión y Veintiséis de Octubre (40, 41, 65);
  Chulucanas y Tambo Grande (21, 44); Castilla y Cura Mori (36, 38); y uno de sierra, San Miguel de El Faique y
  Sondorillo (16, 18; 1 202 y 2 075 m).
- **4 distritos que no comparten celda:** dos de costa, Paita (33) y Sechura (49, a 1 m), y dos de sierra,
  Huancabamba (13, 2 904 m) y Ayabaca (1, 2 066 m).

`download_all` sigue sola: si el API devuelve el límite por hora, imprime la hora a la que reanuda, espera 61 minutos
y continúa; si devuelve el límite diario, se detiene y dice cuántos quedan (vuelvan a correr la celda al día
siguiente: reanuda desde el caché). Si alguna variable llega vacía (la documentación sugiere que ERA5-Land podría no
traer la radiación), la celda la guarda igual y el notebook de comparación lo muestra como nulos.


In [ ]:
from denguekay.data.ingestion.openmeteo import download_all

MUESTRA_ERA5_LAND = [43, 51, 55, 40, 41, 65, 21, 44, 36, 38, 16, 18, 33, 49, 13, 1]

# A2 · Una sola corrida: espera sola tras el límite por hora y se detiene en el límite diario
quedan_era5_land = download_all(
    distritos[distritos.id_distrito.isin(MUESTRA_ERA5_LAND)],
    models="era5_land", cache_dir=CACHE_METEO_ERA5_LAND_2017_2025,
)


## Estudio 06 · 1b′ · ERA5-Land 2000-2025, todos los distritos (humedad y temperatura)

ERA5-Land no trae precipitación, lluvia, viento ni radiación (documentación de Open-Meteo), así que solo se piden
`ERA5_LAND_DAILY_VARS`: las tres temperaturas y la humedad relativa. Las dos celdas usan `download_all`: esperan solas
tras el límite por hora y se detienen en el límite diario; al volver a correrlas siguen desde el caché.

- **A3 · 2017-2025:** completa el caché de A2 con los 49 distritos que faltan (los 16 de la muestra se saltan).
- **B2 · 2000-2016:** caché nuevo `data/bronze/meteo/cache_meteo_era5_land_2000_2016/`.

Son unas 40 000 llamadas según cómo cuenta el API (más de 2 semanas por petición = varias llamadas); con el tope de
10 000 por día, unos 4 a 5 días. Antes de B2, prueben en el navegador que la humedad de ERA5-Land llega con datos en
2000.


In [ ]:
from denguekay.config.paths import CACHE_METEO_ERA5_LAND_2000_2016, CACHE_METEO_ERA5_LAND_2017_2025
from denguekay.data.ingestion.openmeteo import ERA5_LAND_DAILY_VARS, download_all

# A3 · ERA5-Land 2017-2025, los 65 distritos
quedan_era5_land_65 = download_all(
    distritos, models="era5_land", cache_dir=CACHE_METEO_ERA5_LAND_2017_2025, daily_vars=ERA5_LAND_DAILY_VARS,
)


In [ ]:
# B2 · ERA5-Land 2000-2016, los 65 distritos
quedan_era5_land_historia = download_all(
    distritos, start_date="2000-01-01", end_date="2016-12-31",
    models="era5_land", cache_dir=CACHE_METEO_ERA5_LAND_2000_2016, daily_vars=ERA5_LAND_DAILY_VARS,
)


In [ ]:
# B · Solo después de la decisión de Rosa. "best_match" = sin fijar modelo (como 2017-2025).
MODELO_HISTORIA = "POR_DECIDIR"
if MODELO_HISTORIA == "POR_DECIDIR":
    raise RuntimeError("Falta la decisión de Rosa sobre el modelo de la historia 2000-2016.")

quedan_historia = download_pending_batch(
    distritos, start_date="2000-01-01", end_date="2016-12-31",
    models=None if MODELO_HISTORIA == "best_match" else MODELO_HISTORIA,
    cache_dir=CACHE_METEO_2000_2016,
)
